# Used car price regression
Train on `train.csv`, predict `price` for `test.csv` → `submission.csv`.

Approach: feature engineering + CatBoost on **log(price)** (price is heavily right-skewed), validated with 5-fold CV. CatBoost beat LightGBM in CV (RMSLE 0.31 vs 0.49), so it is the final model.

In [1]:
import re, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)
train.head()

(3207, 14) (802, 13)


,id,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,engine_displacement_l,engine_cylinders,transmission_type
0,1,Lexus,IS 300 Base,2018,"50,992 mi.",Gasoline,White,Gray,At least 1 accident or damage reported,Yes,"$28,000",3.5,6.0,Automatic
1,2,Chevrolet,Impala Base,2004,"64,500 mi.",Gasoline,Beige,Beige,None reported,Yes,"$5,900",3.4,6.0,Automatic
2,3,RAM,2500 SLT,2017,"86,000 mi.",Diesel,Gray,Gray,At least 1 accident or damage reported,Yes,"$41,000",6.7,6.0,Automatic
3,4,Mercedes-Benz,SL-Class SL 550,2013,"24,933 mi.",Gasoline,Silver,Red,At least 1 accident or damage reported,Yes,"$40,250",4.6,8.0,Other
4,5,Ford,Shelby GT350R Base,2018,"18,500 mi.",Gasoline,Blue,Black,At least 1 accident or damage reported,Yes,"$77,999",5.2,8.0,Manual


In [2]:
train['model_year'].value_counts()

model_year
2022    295
2021    271
2020    263
2018    243
2019    234
2016    214
2017    212
2015    182
2023    170
2014    141
2013    133
2012    122
2011    102
2008     84
2007     80
2010     75
2005     64
2009     56
2006     55
2004     44
2003     39
2001     26
2002     23
2000     15
1999     14
1998     10
1993      8
1996      8
2024      6
1994      6
1997      6
1995      4
1974      1
1992      1
Name: count, dtype: int64

## Target
Price is a string like `"$28,000"` with a long tail (max ≈ $2.95M), so we train on `log1p(price)`.

In [3]:
price = train['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
y = np.log1p(price)
price.describe()

count    3.207000e+03
mean     4.523930e+04
std      8.486586e+04
min      2.000000e+03
25%      1.700000e+04
50%      3.100000e+04
75%      4.990000e+04
max      2.954083e+06
Name: price, dtype: float64

## Feature engineering

In [4]:
KEYWORDS = ['amg', 'm3', 'm4', 'm5', 'gt', 'turbo', 'sport', 'premium', 'limited', 'platinum', 'base',
            'hybrid', '4matic', 'awd', 'xdrive', 'quattro', 'rs', 'srt', 'hellcat', 'trackhawk', 'raptor', 'denali']
COLOR_RE = r'(black|white|gray|grey|silver|blue|red|green|brown|beige|gold|orange|yellow|purple|tan)'
REF_YEAR = 2026

def make_features(df):
    d = df.copy()
    d['milage'] = d['milage'].str.replace(r'[^\d]', '', regex=True).astype(float)
    d['age'] = REF_YEAR - d['model_year']
    d['mi_per_yr'] = d['milage'] / (d['age'] + 1)
    # missing fuel + missing engine size is almost always an EV
    d['is_ev'] = (d['fuel_type'].isna() & d['engine_displacement_l'].isna()).astype(int)
    d['fuel_type'] = d['fuel_type'].replace({'–': 'Unknown', 'not supported': 'Unknown'}).fillna('Missing')
    d['accident'] = (d['accident'] == 'At least 1 accident or damage reported').astype(int)
    d['clean_title'] = (d['clean_title'] == 'Yes').astype(int)
    # ~30% of test models are unseen in train, so add coarser model groupings
    d['base_model'] = d['model'].str.split().str[0]
    d['brand_base'] = d['brand'] + '_' + d['base_model']
    for c in ['ext_col', 'int_col']:
        d[c + '_s'] = (d[c].str.lower().str.extract(COLOR_RE, expand=False)
                         .fillna('other').replace({'grey': 'gray'}))
    m = d['model'].str.lower()
    for k in KEYWORDS:
        d['kw_' + k] = m.str.contains(r'\b' + k + r'\b').astype(int)
    return d.drop(columns=['id', 'price'], errors='ignore')

X = make_features(train)
X_test = make_features(test)
CATS = ['brand', 'model', 'fuel_type', 'ext_col', 'int_col', 'transmission_type',
        'base_model', 'brand_base', 'ext_col_s', 'int_col_s']
X.head()

,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,engine_displacement_l,...,kw_4matic,kw_awd,kw_xdrive,kw_quattro,kw_rs,kw_srt,kw_hellcat,kw_trackhawk,kw_raptor,kw_denali
0,Lexus,IS 300 Base,2018,50992.0,Gasoline,White,Gray,1,1,3.5,...,0,0,0,0,0,0,0,0,0,0
1,Chevrolet,Impala Base,2004,64500.0,Gasoline,Beige,Beige,0,1,3.4,...,0,0,0,0,0,0,0,0,0,0
2,RAM,2500 SLT,2017,86000.0,Diesel,Gray,Gray,1,1,6.7,...,0,0,0,0,0,0,0,0,0,0
3,Mercedes-Benz,SL-Class SL 550,2013,24933.0,Gasoline,Silver,Red,1,1,4.6,...,0,0,0,0,0,0,0,0,0,0
4,Ford,Shelby GT350R Base,2018,18500.0,Gasoline,Blue,Black,1,1,5.2,...,0,0,0,0,0,0,0,0,0,0


## 5-fold cross-validation

In [5]:
PARAMS = dict(iterations=2000, learning_rate=0.03, depth=6, loss_function='RMSE', verbose=0)
SEEDS = [0, 1, 2]
rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2))

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
kf = KFold(5, shuffle=True, random_state=42)
for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    for seed in SEEDS:
        m = CatBoostRegressor(**PARAMS, random_seed=seed)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CATS)
        oof[va_idx] += m.predict(X.iloc[va_idx]) / len(SEEDS)
        test_pred += m.predict(X_test) / (len(SEEDS) * kf.n_splits)
    print(f'fold {fold}: RMSLE {rmse(oof[va_idx], y.iloc[va_idx]):.4f}')

print(f'\nCV RMSLE : {rmse(oof, y):.4f}')
print(f'CV R² (log): {1 - np.var(oof - y) / np.var(y):.4f}')
print(f'CV RMSE ($): {rmse(np.expm1(oof), price):,.0f}')
print(f'CV MAE  ($): {np.mean(np.abs(np.expm1(oof) - price)):,.0f}')
print(f'CV MAPE    : {np.mean(np.abs(np.expm1(oof) - price) / price):.1%}')

fold 0: RMSLE 0.3474


fold 1: RMSLE 0.2911


fold 2: RMSLE 0.2971


fold 3: RMSLE 0.3091


fold 4: RMSLE 0.2950

CV RMSLE : 0.3086
CV R² (log): 0.8692
CV RMSE ($): 70,468
CV MAE  ($): 11,115
CV MAPE    : 21.2%


## Feature importance (last fold model)

In [6]:
pd.Series(m.get_feature_importance(), index=X.columns).sort_values(ascending=False).head(15)

milage                   19.127155
engine_displacement_l    13.687829
brand                    12.938935
age                      11.477826
model_year                9.243716
base_model                7.444281
mi_per_yr                 5.456124
engine_cylinders          3.544882
fuel_type                 2.579289
int_col_s                 2.429783
transmission_type         1.984899
model                     1.746121
kw_base                   1.598275
brand_base                1.480564
ext_col_s                 1.195430
dtype: float64

## Predict test and save
Test predictions are the average of the 15 fold×seed models, transformed back from log scale.

In [7]:
submission = pd.DataFrame({'id': test['id'], 'price': np.expm1(test_pred).round(0)})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.describe()

(802, 2)


,id,price
count,802.000000,802.000000
mean,3608.500000,39951.332918
std,231.661751,41517.367763
min,3208.000000,4525.000000
25%,3408.250000,16679.750000
50%,3608.500000,31951.000000
75%,3808.750000,50422.750000
max,4009.000000,643460.000000
